In [23]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv('../data/ufc_clean.csv')
print(df.shape)
print(df.columns.tolist())

(5246, 71)
['RedOdds', 'BlueOdds', 'Winner', 'TitleBout', 'WeightClass', 'Gender', 'NumberOfRounds', 'BlueCurrentLoseStreak', 'BlueCurrentWinStreak', 'BlueDraws', 'BlueAvgSigStrLanded', 'BlueAvgSigStrPct', 'BlueAvgSubAtt', 'BlueAvgTDLanded', 'BlueAvgTDPct', 'BlueLongestWinStreak', 'BlueLosses', 'BlueTotalRoundsFought', 'BlueTotalTitleBouts', 'BlueWinsByDecisionMajority', 'BlueWinsByDecisionSplit', 'BlueWinsByDecisionUnanimous', 'BlueWinsByKO', 'BlueWinsBySubmission', 'BlueWinsByTKODoctorStoppage', 'BlueWins', 'BlueStance', 'BlueHeightCms', 'BlueReachCms', 'BlueWeightLbs', 'RedCurrentLoseStreak', 'RedCurrentWinStreak', 'RedDraws', 'RedAvgSigStrLanded', 'RedAvgSigStrPct', 'RedAvgSubAtt', 'RedAvgTDLanded', 'RedAvgTDPct', 'RedLongestWinStreak', 'RedLosses', 'RedTotalRoundsFought', 'RedTotalTitleBouts', 'RedWinsByDecisionMajority', 'RedWinsByDecisionSplit', 'RedWinsByDecisionUnanimous', 'RedWinsByKO', 'RedWinsBySubmission', 'RedWinsByTKODoctorStoppage', 'RedWins', 'RedStance', 'RedHeightCms

In [24]:
# De kolommen die we als input meegeven aan het model
features = [
    'LoseStreakDif', 'WinStreakDif', 'LongestWinStreakDif',
    'WinDif', 'LossDif', 'TotalRoundDif', 'TotalTitleBoutDif',
    'KODif', 'SubDif', 'HeightDif', 'ReachDif', 'AgeDif',
    'SigStrDif', 'AvgSubAttDif', 'AvgTDDif', 'RedOdds', 'BlueOdds'
]

X = df[features]         # input: de statistieken
y = df['Winner']         # output: wie wint

print(X.shape)
print(y.value_counts())  # hoeveel Red wins vs Blue wins

(5246, 17)
Winner
Red     3027
Blue    2219
Name: count, dtype: int64


In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Traindata:  {X_train.shape}")
print(f"Testdata:   {X_test.shape}")

Traindata:  (4196, 17)
Testdata:   (1050, 17)


In [26]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("Model getraind!")

Model getraind!


In [27]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuraatheid: {accuracy:.2%}")

Accuraatheid: 68.67%


In [28]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# XGBoost heeft nummers nodig, geen tekst (Red/Blue → 0/1)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Opnieuw splitsen met de encoded labels
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)

# Model trainen
xgb_model = XGBClassifier(n_estimators=100, random_state=42)
xgb_model.fit(X_train2, y_train2)

# Testen
y_pred2 = xgb_model.predict(X_test2)
accuracy2 = accuracy_score(y_test2, y_pred2)
print(f"XGBoost accuraatheid: {accuracy2:.2%}")

XGBoost accuraatheid: 65.33%


In [38]:
features_v2 = [
    # Dif kolommen (hadden we al)
    'LoseStreakDif', 'WinStreakDif', 'LongestWinStreakDif',
    'WinDif', 'LossDif', 'TotalRoundDif', 'TotalTitleBoutDif',
    'KODif', 'SubDif', 'HeightDif', 'ReachDif', 'AgeDif',
    'SigStrDif', 'AvgSubAttDif', 'AvgTDDif',

    # Individuele statistieken Red
    'RedCurrentWinStreak', 'RedCurrentLoseStreak', 'RedAvgSigStrLanded',
    'RedAvgSigStrPct', 'RedAvgTDLanded', 'RedAvgTDPct',
    'RedWins', 'RedLosses', 'RedAge', 'RedHeightCms', 'RedReachCms',

    # Individuele statistieken Blue
    'BlueCurrentWinStreak', 'BlueCurrentLoseStreak', 'BlueAvgSigStrLanded',
    'BlueAvgSigStrPct', 'BlueAvgTDLanded', 'BlueAvgTDPct',
    'BlueWins', 'BlueLosses', 'BlueAge', 'BlueHeightCms', 'BlueReachCms',

    # Extra context
    'NumberOfRounds',

    # Odds
    'RedOdds', 'BlueOdds',

]

X2 = df[features_v2]
X_train3, X_test3, y_train3, y_test3 = train_test_split(
    X2, y, test_size=0.2, random_state=42
)

model_v2 = RandomForestClassifier(n_estimators=100, random_state=42)
model_v2.fit(X_train3, y_train3)

y_pred3 = model_v2.predict(X_test3)
accuracy3 = accuracy_score(y_test3, y_pred3)
print(f"Random Forest v2 accuraatheid: {accuracy3:.2%}")

Random Forest v2 accuraatheid: 66.48%


In [39]:
# BetterRank omzetten naar nummers
df['BetterRank_encoded'] = df['BetterRank'].map({'Red': 1, 'Blue': -1, 'neither': 0})

features_v3 = features_v2 + ['BetterRank_encoded']

X3 = df[features_v3]
X_train4, X_test4, y_train4, y_test4 = train_test_split(
    X3, y, test_size=0.2, random_state=42
)

model_v3 = RandomForestClassifier(n_estimators=300, random_state=42)
model_v3.fit(X_train4, y_train4)

y_pred4 = model_v3.predict(X_test4)
accuracy4 = accuracy_score(y_test4, y_pred4)
print(f"Random Forest v3 accuraatheid: {accuracy4:.2%}")

Random Forest v3 accuraatheid: 68.29%


In [32]:
import matplotlib.pyplot as plt

importances = model_v2.feature_importances_
feature_names = features_v2

feat_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
feat_df = feat_df.sort_values('importance', ascending=False)

print(feat_df.to_string())

                  feature  importance
12              SigStrDif    0.048114
17     RedAvgSigStrLanded    0.045934
14               AvgTDDif    0.045525
18        RedAvgSigStrPct    0.044169
28    BlueAvgSigStrLanded    0.044102
29       BlueAvgSigStrPct    0.041219
13           AvgSubAttDif    0.038344
19         RedAvgTDLanded    0.036407
34                BlueAge    0.035808
23                 RedAge    0.035805
11                 AgeDif    0.033755
20            RedAvgTDPct    0.033625
5           TotalRoundDif    0.032753
30        BlueAvgTDLanded    0.032523
31           BlueAvgTDPct    0.031381
36           BlueReachCms    0.026945
25            RedReachCms    0.026353
10               ReachDif    0.025931
9               HeightDif    0.025062
35          BlueHeightCms    0.023138
3                  WinDif    0.022702
24           RedHeightCms    0.022663
4                 LossDif    0.021975
21                RedWins    0.021116
7                   KODif    0.020011
32          

In [40]:
features_v4 = [f for f in features_v2 if f != 'NumberOfRounds']

X4 = df[features_v4]
X_train5, X_test5, y_train5, y_test5 = train_test_split(
    X4, y, test_size=0.2, random_state=42
)

model_v4 = RandomForestClassifier(n_estimators=100, random_state=42)
model_v4.fit(X_train5, y_train5)

y_pred5 = model_v4.predict(X_test5)
accuracy5 = accuracy_score(y_test5, y_pred5)
print(f"Random Forest v4 accuraatheid: {accuracy5:.2%}")

Random Forest v4 accuraatheid: 66.38%


In [34]:
odds_kolommen = [col for col in df.columns if 'odds' in col.lower()]
print(odds_kolommen)

df[['RedOdds', 'BlueOdds']].describe()

['RedOdds', 'BlueOdds']


,RedOdds,BlueOdds
count,5246.000000,5246.000000
mean,-114.553755,58.378574
std,274.526345,249.835915
min,-2100.000000,-1100.000000
25%,-250.000000,-150.000000
50%,-147.000000,130.000000
75%,130.000000,210.000000
max,700.000000,1150.000000
